# Лекция 1. Знакомство с обучением с подкреплением

**Цели лекции**

1. Выделить основные отличительные черты Reinforcement Learning.
2. Понимать области применения Reinforcement Learning.
3. Разобраться, что отличает Reinforcement Learning от других областей машинного обучения.

**План**

1. Как устроен курс
2. Что такое Reinforcement Learning
3. Примеры: Maze, Frozen Lake, Atari, CartPole, гуманоид
4. Определите сами: состояние, действие, награда в задачах из жизни
5. Живое демо: агент в среде Gymnasium
6. Чем RL отличается от других видов машинного обучения
7. Случайные процессы, марковские цепи и MDP
8. Награда как формулировка задачи
9. Exploration vs exploitation: многорукие бандиты
10. Первый алгоритм: метод Cross-Entropy
11. Где применяется RL
12. Карта курса, итоги, литература

## 1. Как устроен курс

* **16 недель**, одно занятие в неделю: лекция + семинар. Каждая неделя — папка `NN-topic-name/` в репозитории с тремя ноутбуками: `lecture/`, `seminar/`, `homework/`.
* **Домашние задания** почти каждую неделю, в `.ipynb`. Часть проверок — через `assert`, так что можно проверить себя до сдачи.
* **Оценка** (черновик, обсуждаем сегодня): 60% домашние задания, 30% итоговый проект, 10% активность на семинарах.
* **Итоговый проект**: своя реализация RL-агента или мини-исследование на среде по выбору. Темы выбираем на неделе 14, защита на неделе 16.
* **Инструменты**: Python 3.10+, [Gymnasium](https://gymnasium.farama.org/) (среды), [PyTorch](https://pytorch.org/) (нейросети), Jupyter.
  Библиотеки [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) и [CleanRL](https://github.com/vwxyzjn/cleanrl) читаем как справочник, но алгоритмы в домашках пишем сами.
* **Что нужно уметь на входе**: линейная алгебра, теория вероятностей, градиентный спуск, Python с numpy. Нейросети и PyTorch — желательно, но необходимый минимум разберём на мини-семинаре `../seminar/pytorch_intro.ipynb`.

Установка окружения:

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

## 2. Что такое Reinforcement Learning

Начнём с картинки, которую все понимают без определений: дрессировщик и собака.

<img src="../../assets/ref/agent_env_dog.png" width="420">

Собака **наблюдает** (жест, голос), **действует** (садится, лает, убегает), а дрессировщик выдаёт **награду** (лакомство есть или нет). Никто не объясняет собаке, *как* правильно, есть только последствия. Через десяток повторений действие, за которым идёт лакомство, случается чаще. Это и есть обучение с подкреплением.

Та же схема в общем виде:

<img src="../../assets/ref/agent_env_brain.png" width="420">

* **Агент** — тот, кто принимает решения (мозг, программа). Единственное, что мы обучаем.
* **Среда** — всё остальное: физический мир, игра, биржа, другой игрок.
* На каждом шаге $t$ агент видит **состояние** $S_t$ (или наблюдение), выбирает **действие** $A_t$, среда отвечает **наградой** $R_t$ и новым состоянием $S_{t+1}$.

**Цель агента** — максимизировать суммарную награду

$$
G = \sum_{t=0}^{\infty} \gamma^t R_t, \qquad \gamma \in [0, 1].
$$

**Reinforcement Learning (RL)** — раздел машинного обучения о том, как агент, взаимодействуя со средой, учится выбирать действия так, чтобы суммарная награда была как можно больше.

<small>Иллюстрации в этом и следующем разделах — из лекции 1 курса [imm-rl-lab](https://github.com/imm-rl-lab/reinforcement_learning_course) (А. Плаксин).</small>

### Откуда слово «подкрепление»

Термин пришёл из психологии. **Закон эффекта** Торндайка (1911): действия с приятными последствиями в той же ситуации повторяются чаще. **Подкрепление** (reinforcement) у Скиннера (1938): всё, что увеличивает частоту поведения. Математику добавила теория оптимального управления: **Беллман** (1957) ввёл ценность состояния и уравнение, которое мы сегодня увидим. **Саттон и Барто** в 1980-х соединили обе линии в то, что теперь называется RL.

![origins](../../assets/rl_origins.png)

## 3. Примеры

Для каждой задачи нужно ответить на три вопроса: **что агент видит** (состояния), **что он может делать** (действия), **за что ему платят** (награда). Начнём с самого простого.

### Maze

<img src="../../assets/ref/maze.png" width="300">

* **Состояния**: белые клетки лабиринта. Агент всегда знает, в какой клетке он стоит.
* **Действия**: ↑, →, ↓, ←. Шаг в стену оставляет агента на месте.
* **Награда**: −1 на каждом шаге, 0 если стоять в Goal. Чем быстрее дошёл, тем меньше штрафа.
* **Эпизод**: от Start до Goal (или до лимита шагов).

Обратите внимание: агенту никто не говорит «иди направо». Он узнаёт, что путь хороший, только по сумме штрафов в конце. Это отличие RL от обучения с учителем мы разберём в разделе 6.

### Frozen Lake

<img src="../../assets/ref/frozen_lake.png" width="520">

* **Состояния**: 16 клеток озера 4×4. `S` старт, `F` лёд, `H` прорубь, `G` цель.
* **Действия**: влево, вниз, вправо, вверх.
* **Награда**: +1 за достижение `G`, 0 во всех остальных случаях. Попал в прорубь — эпизод окончен без награды.
* **Особенность**: лёд скользкий. Агент идёт туда, куда хотел, лишь с вероятностью 1/3, иначе его сносит вбок. Одно и то же действие в одном и том же состоянии может привести в разные клетки. Это первый пример **случайной среды**.

С Frozen Lake будем работать на семинаре.

### Atari Games

<img src="../../assets/ref/atari.png" width="300">

* **Состояния**: пиксели с экрана (210×160×3 чисел). Агент видит то же, что и человек.
* **Действия**: →, ←, «0» (ничего не делать), огонь и их комбинации, всего до 18.
* **Награда**: очки в игре.

В 2015 году одна и та же нейросеть (DQN, неделя 6) научилась играть в 49 игр Atari, глядя только на пиксели и счёт. Это событие сделало RL знаменитым.

### CartPole

<img src="../../assets/ref/cartpole.png" width="380">

* **Состояния**: $\mathbb{R}^4$ — положение и скорость тележки, угол и угловая скорость шеста. Или пиксели с экрана.
* **Действия**: толкнуть тележку → или ←.
* **Награда**: +1 на каждом шаге, пока шест стоит; эпизод заканчивается, когда шест упал или тележка уехала за край.

Задача простая, но в ней впервые появляется **непрерывное пространство состояний**: таблицу «состояние → действие» уже не составить. Через минуту запустим её вживую.

### Гуманоид (MuJoCo)

<img src="../../assets/ref/humanoid.png" width="300">

* **Состояния**: $\mathbb{R}^{26}$ и больше — углы и скорости всех суставов, положение центра масс.
* **Действия**: $\mathbb{R}^{6}$ … $\mathbb{R}^{17}$ — усилия в каждом суставе. **Действия непрерывные**: не «влево / вправо», а число.
* **Награда**: +1 за каждый момент времени, пока робот не упал, плюс бонус за скорость движения вперёд.

Такими задачами занимаются методы для непрерывного управления (недели 10 и дальше). Их же используют, чтобы учить ходить настоящих четвероногих и двуногих роботов.

## 4. Определите сами

Умение **сформулировать** задачу на языке RL важнее знания конкретных алгоритмов: неправильно выбранная награда или состояние испортят любой метод. Потренируемся. Для каждой задачи назовите состояние, действия, награду, эпизод и найдите, где в среде случайность. Ответы спрятаны, сначала подумайте сами.

### Шахматы

<details>
<summary>Ответ</summary>

* **Состояние**: расположение фигур на доске, чей ход, право на рокировку. Всё видно, скрытой информации нет.
* **Действия**: любой допустимый ход. Множество действий зависит от состояния.
* **Награда**: +1 победа, −1 поражение, 0 ничья, всё остальное время 0. Награда приходит **только в конце**, через десятки ходов.
* **Эпизод**: партия.
* **Случайность**: в правилах её нет, но есть соперник. С точки зрения агента ход соперника — это случайный отклик среды.
</details>

### Такси (Gymnasium `Taxi-v3`)

Такси ездит по сетке 5×5, нужно забрать пассажира в одной из 4 точек и отвезти в другую.

<details>
<summary>Ответ</summary>

* **Состояние**: позиция такси (25 вариантов) × где пассажир (4 точки или в машине) × куда ехать (4 точки) = 500 состояний.
* **Действия**: 4 направления, посадить, высадить.
* **Награда**: −1 за каждый шаг, +20 за успешную высадку, −10 за попытку посадить/высадить не там.
* **Эпизод**: от появления пассажира до высадки.
* **Случайность**: только в начальном состоянии (где появится пассажир и куда ему нужно). Сами переходы детерминированные.
</details>

### Охлаждение дата-центра

Нужно управлять насосами, чиллерами и вентиляторами так, чтобы серверы не перегревались, а счёт за электричество был минимальным.

<details>
<summary>Ответ</summary>

* **Состояние**: показания сотен датчиков: температуры, нагрузка серверов, погода снаружи, текущие уставки оборудования. Агент видит не всё (например, не знает нагрузку через час), это **наблюдение**, а не полное состояние.
* **Действия**: уставки оборудования: обороты насосов, температура воды. Действия **непрерывные**.
* **Награда**: минус потреблённая энергия за шаг, большой штраф за выход температуры за допустимый диапазон.
* **Эпизод**: задача **непрерывная**, естественного конца нет. Удобно резать на дни.
* **Случайность**: погода, нагрузка на серверы, износ оборудования.
</details>

### Торговый агент

Программа управляет портфелем из нескольких акций.

<details>
<summary>Ответ</summary>

* **Состояние**: история цен и объёмов, текущая позиция, остаток денег. Настоящего «состояния рынка» агент не знает, это снова наблюдение.
* **Действия**: купить / продать / держать по каждой бумаге, или доля портфеля в каждой.
* **Награда**: изменение стоимости портфеля за шаг с учётом комиссий; часто ещё штраф за риск.
* **Эпизод**: торговый день или месяц.
* **Случайность**: почти всё. Отклик среды (цена завтра) зависит от миллионов других участников, и правила меняются со временем. Поэтому это одна из самых трудных задач для RL.
</details>

Шпаргалка, как узнать термины в новой задаче:

| Термин | Вопрос, который нужно задать |
|---|---|
| Состояние / наблюдение | Что агент знает в момент принятия решения? Чего он не знает? |
| Действие | Что агент может изменить прямо сейчас? Дискретный выбор или число? |
| Награда | Какое **одно число** за шаг измеряет успех? Приходит сразу или в конце? |
| Эпизод | Есть ли естественный конец? Или задача бесконечная? |
| Случайность среды | Может ли одно и то же действие в одном и том же состоянии привести к разным исходам? |

## 5. Живое демо: агент в среде Gymnasium

[Gymnasium](https://gymnasium.farama.org/) — стандартный интерфейс к средам: `env.reset()` возвращает первое наблюдение, `env.step(action)` — следующее наблюдение, награду и флаги окончания эпизода. Все среды курса будут выглядеть так.

Запустим CartPole.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)
print("observation_space:", env.observation_space)
print("action_space:     ", env.action_space)
print("первое наблюдение:", obs)

obs, reward, terminated, truncated, info = env.step(1)  # 1 = толкнуть вправо
print("после шага:        ", obs, "reward =", reward, "done =", terminated or truncated)

In [ ]:
# Эпизод со случайной политикой: сохраним кадры.
env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, _ = env.reset(seed=1)
frames = []
for t in range(60):
    frames.append(env.render())
    obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
    if terminated or truncated:
        break
env.close()

idx = np.linspace(0, len(frames) - 1, 6).astype(int)
fig, axes = plt.subplots(1, 6, figsize=(16, 2.6))
for ax, i in zip(axes, idx):
    ax.imshow(frames[i])
    ax.set_title(f"t = {i}")
    ax.axis("off")
plt.suptitle(f"Случайная политика: шест падает за {len(frames)} шагов")
plt.show()

Случайная политика держит шест около 20 шагов. Правило выбора действия по состоянию называется **политикой**. Напишем политику руками: если шест падает вправо (угловая скорость положительная), толкаем тележку вправо, и наоборот.

In [ ]:
def random_policy(obs):
    return np.random.randint(2)

def heuristic_policy(obs):
    x, x_dot, theta, theta_dot = obs
    return int(theta_dot > 0)   # толкаем в ту сторону, куда падает шест

def run_episodes(policy, n_episodes=50, seed=0):
    env = gym.make("CartPole-v1")
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total = 0
        while True:
            obs, r, terminated, truncated, _ = env.step(policy(obs))
            total += r
            if terminated or truncated:
                break
        returns.append(total)
    env.close()
    return np.array(returns)

np.random.seed(0)
for name, policy in [("случайная", random_policy), ("эвристика", heuristic_policy)]:
    G = run_episodes(policy)
    print(f"{name:10s}: средняя суммарная награда {G.mean():6.1f} ± {G.std():5.1f} (максимум 500)")

Эвристика в 10 раз лучше случайной, но её придумал человек, зная физику задачи, и до максимума она не дотягивает. Для Atari, шахмат или гуманоида с 17 суставами такое правило руками не напишешь.

**Задача RL** — получить политику не хуже эвристики, **не зная** устройства среды, только из опыта взаимодействия. Первый такой алгоритм увидим сегодня в разделе 10, а на неделе 6 нейросеть будет стабильно держать 500 шагов.

Если установлен `box2d`, можно посмотреть среду посложнее: LunarLander.

In [ ]:
try:
    env = gym.make("LunarLander-v3", render_mode="rgb_array")
    obs, _ = env.reset(seed=0)
    frames = []
    for t in range(120):
        frames.append(env.render())
        obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
        if terminated or truncated:
            break
    env.close()
    idx = np.linspace(0, len(frames) - 1, 4).astype(int)
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
    for ax, i in zip(axes, idx):
        ax.imshow(frames[i]); ax.set_title(f"t = {i}"); ax.axis("off")
    plt.suptitle("LunarLander, случайная политика: наблюдение из 8 чисел, 4 действия (двигатели)")
    plt.show()
except Exception as e:  # box2d не установлен или не собрался
    print("LunarLander недоступен:", type(e).__name__, "-", str(e)[:120])
    print("Гифку можно посмотреть в документации: https://gymnasium.farama.org/environments/box2d/lunar_lander/")

## 6. Чем RL отличается от других видов машинного обучения

![paradigms](../../assets/ml_paradigms.png)

| | Supervised Learning | Unsupervised Learning | Reinforcement Learning |
|---|---|---|---|
| Что дано | размеченные пары (x, y) | неразмеченные x | среда, с которой можно взаимодействовать |
| Обратная связь | правильный ответ сразу | нет обратной связи | награда: число, часто с задержкой |
| Откуда данные | собраны заранее, фиксированы | собраны заранее, фиксированы | порождает сам агент своими действиями |
| Цель | предсказывать y | найти структуру в данных | максимизировать суммарную награду |
| Пример | «кошка или собака на фото?» | «на какие группы делятся клиенты?» | «как пройти уровень?» |

**Четыре отличительные черты RL**

1. **Нет правильного ответа, есть оценка.** В supervised learning учитель говорит «здесь должно быть 7». В RL никто не говорит «здесь нужно было повернуть налево», есть только число.
2. **Отложенная награда.** В шахматах награда приходит через 40 ходов после решающей ошибки. Какое из 40 действий виновато? Это задача **credit assignment**.
3. **Данные порождает сам агент.** Плохая политика видит только плохие состояния. Данные не независимы и меняются по ходу обучения; гарантии статистики для i.i.d. выборок не работают.
4. **Нужно исследовать.** Чтобы найти лучшее поведение, надо пробовать новое, а новое в среднем невыгодно. В supervised learning такой проблемы нет вообще (раздел 9).

Общее с остальным ML — статистика и нейросети как инструмент. Отличается сама **постановка задачи**: не «предскажи», а «действуй».

## 7. Случайные процессы, марковские цепи и MDP

Чтобы что-то доказывать и программировать, нужна математическая модель среды. Построим её в три шага: марковская цепь → цепь с наградами → процесс принятия решений.

### 7.1 Случайный процесс и марковское свойство

**Случайный процесс** — последовательность случайных величин $S_0, S_1, S_2, \ldots$: состояние среды в моменты времени $0, 1, 2, \ldots$. Погода по дням, курс акции по минутам, клетка, в которой стоит агент.

В общем случае $S_{t+1}$ может зависеть от всей истории. **Марковское свойство** — упрощение, на котором держится весь RL:

$$
\mathbb{P}[S_{t+1} \mid S_t] = \mathbb{P}[S_{t+1} \mid S_0, S_1, \ldots, S_t].
$$

«Будущее зависит от прошлого только через настоящее». Если состояние выбрано правильно, вся полезная история уже в нём: в CartPole поэтому в наблюдении есть скорости, а в Atari складывают 4 последних кадра.

**Марковская цепь** — набор состояний $\mathcal{S}$ и матрица переходов $P$, где $P_{ss'} = \mathbb{P}[S_{t+1} = s' \mid S_t = s]$. Строки матрицы суммируются в 1.

Пример: день студента.

![chain](../../assets/markov_chain.png)

In [ ]:
states = ["Лекция", "Соцсети", "Сон", "Экзамен сдан"]
P = np.array([
    [0.0, 0.5, 0.2, 0.3],   # из Лекции
    [0.3, 0.0, 0.7, 0.0],   # из Соцсетей
    [1.0, 0.0, 0.0, 0.0],   # из Сна
    [0.0, 0.0, 0.0, 1.0],   # Экзамен сдан: терминальное, остаёмся навсегда
])
assert np.allclose(P.sum(axis=1), 1.0)

rng = np.random.default_rng(0)

def sample_chain(P, s0=0, max_steps=20):
    s, path = s0, [s0]
    for _ in range(max_steps):
        s = rng.choice(len(P), p=P[s])
        path.append(s)
        if P[s, s] == 1.0:      # терминальное состояние
            break
    return path

for _ in range(4):
    print(" -> ".join(states[s] for s in sample_chain(P)))

# Распределение через n шагов: p_n = p_0 P^n
p = np.array([1.0, 0, 0, 0])
for n in [1, 2, 5, 20]:
    print(f"через {n:2d} шагов: " + ", ".join(f"{states[i]} {x:.2f}" for i, x in enumerate(p @ np.linalg.matrix_power(P, n))))

Матрица переходов отвечает на любой вопрос о будущем цепи: где будем через $n$ шагов ($p_0 P^n$), с какой вероятностью когда-нибудь сдадим экзамен, сколько в среднем это займёт. Никакой истории хранить не нужно.

### 7.2 Марковский процесс с наградами (MRP)

Добавим к цепи **награду** $R(s)$ за посещение состояния и коэффициент дисконтирования $\gamma$. Теперь у каждой траектории есть **return** — суммарная дисконтированная награда с момента $t$:

$$
G_t = R_{t} + \gamma R_{t+1} + \gamma^2 R_{t+2} + \ldots = \sum_{k=0}^{\infty} \gamma^k R_{t+k}.
$$

Зачем $\gamma < 1$:

* сумма конечна даже для бесконечных траекторий;
* награда через $k$ шагов весит $\gamma^k$: «горизонт планирования» агента $\approx 1/(1-\gamma)$;
* математически удобно (сжимающее отображение, увидим на неделе 3).

In [ ]:
ks = np.arange(0, 200)
for gamma in [0.5, 0.9, 0.99]:
    plt.plot(ks, gamma ** ks, label=f"γ = {gamma}: горизонт ≈ {1/(1-gamma):.0f} шагов")
plt.xlabel("k (через сколько шагов придёт награда)")
plt.ylabel("вес награды γ^k")
plt.title("Дисконтирование: насколько агенту важно будущее")
plt.legend()
plt.show()

**Ценность состояния** — ожидаемый return, если стартовать из $s$:

$$
V(s) = \mathbb{E}[G_t \mid S_t = s].
$$

Ключевое наблюдение: return **рекурсивен**, $G_t = R_t + \gamma G_{t+1}$. Взяв матожидание, получаем **уравнение Беллмана** для MRP:

$$
V(s) = R(s) + \gamma \sum_{s'} P_{ss'} V(s'), \qquad \text{в матричном виде } V = R + \gamma P V .
$$

Это система линейных уравнений, её можно решить явно: $V = (I - \gamma P)^{-1} R$, или просто повторять правую часть, пока не сойдётся. Посчитаем для студента: −2 за лекцию (скучно), +1 за соцсети, 0 за сон, +20 за сданный экзамен.

In [ ]:
R = np.array([-2.0, 1.0, 0.0, 20.0])
gamma = 0.9
P_ = P.copy(); P_[3] = 0.0          # после экзамена награды больше нет: терминальное состояние

V_exact = np.linalg.solve(np.eye(4) - gamma * P_, R)

V = np.zeros(4)
for it in range(200):
    V_new = R + gamma * P_ @ V
    if np.max(np.abs(V_new - V)) < 1e-8:
        break
    V = V_new

for s, v_it, v_ex in zip(states, V, V_exact):
    print(f"V({s:12s}) = {v_it:7.3f}   (точное {v_ex:7.3f})")
print(f"итераций до сходимости: {it}")

Лекция сама по себе неприятна ($R = -2$), но $V(\text{Лекция})$ выше, чем $V(\text{Соцсети})$: из лекции ближе к экзамену. Ценность учитывает **будущее**, а не только текущую награду. Именно ценность, а не награду, будет максимизировать агент.

### 7.3 MDP: добавляем действия

До сих пор состояния менялись сами. Дадим агенту возможность **влиять** на переходы. **Марковский процесс принятия решений** (Markov Decision Process) — кортеж $\langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$:

* $\mathcal{S}$ — множество состояний;
* $\mathcal{A}$ — множество действий;
* $P(s' \mid s, a) = \mathbb{P}[S_{t+1} = s' \mid S_t = s, A_t = a]$ — вероятности переходов, теперь они зависят от действия;
* $R(s, a)$ — ожидаемая награда за действие $a$ в состоянии $s$;
* $\gamma$ — коэффициент дисконтирования.

Марковское свойство теперь читается так: $\mathbb{P}[S_{t+1} \mid S_t, A_t] = \mathbb{P}[S_{t+1} \mid S_0, A_0, \ldots, S_t, A_t]$.

**Политика** $\pi(a \mid s)$ — распределение над действиями в состоянии $s$. Детерминированная политика $\pi: \mathcal{S} \to \mathcal{A}$ — частный случай. Как порождается траектория:

* агент в $S_0$, выбирает $A_0 \sim \pi(\cdot \mid S_0)$;
* получает $R_0 = R(S_0, A_0)$ и переходит в $S_1 \sim P(\cdot \mid S_0, A_0)$;
* выбирает $A_1 \sim \pi(\cdot \mid S_1)$, и так далее.

$$
\tau = (S_0, A_0, S_1, A_1, S_2, A_2, \ldots), \qquad G(\tau) = \sum_{t=0}^{\infty} \gamma^t R(S_t, A_t).
$$

Важный факт: **MDP + фиксированная политика = MRP**. Если политика выбрана, действия можно «усреднить» и получить обычную цепь с матрицей $P^\pi_{ss'} = \sum_a \pi(a \mid s) P(s' \mid s, a)$. Значит, всё, что мы умеем для MRP, переносится.

Пример MDP — GridWorld: состояние — клетка, действие — направление, переходы «скользкие» (это и есть стохастичность среды), награда за выход и за яму.

![gridworld](../../assets/mdp_gridworld.png)

### 7.4 Ценности и уравнения Беллмана для MDP

* **Value function** $V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$: насколько хорошо находиться в $s$, следуя $\pi$.
* **Action-value function** $Q^\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]$: насколько хорошо в $s$ сделать $a$, а дальше следовать $\pi$.

Из рекурсии $G_t = R_t + \gamma G_{t+1}$ получаются **уравнения Беллмана (ожидания)**:

$$
V^\pi(s) = \sum_a \pi(a \mid s) \Big[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a)\, V^\pi(s') \Big],
$$

$$
Q^\pi(s, a) = R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) \sum_{a'} \pi(a' \mid s')\, Q^\pi(s', a').
$$

Связь между ними: $V^\pi(s) = \sum_a \pi(a \mid s)\, Q^\pi(s, a)$.

Для **оптимальной** политики сумма по $a'$ с весами $\pi$ превращается в $\max_{a'}$ — это уравнения оптимальности Беллмана, ими займёмся на неделе 3. Почти все алгоритмы курса так или иначе решают одно из этих уравнений: динамическое программирование — итерациями, TD-обучение — по сэмплам, deep RL — нейросетью, минимизируя невязку.

Маленькая проверка: цепочка из трёх состояний, политика всегда идёт вправо, переход детерминированный, +1 за приход в терминальное $s_2$. Тогда $V^\pi(s_1) = 1$, $V^\pi(s_0) = \gamma$.

In [ ]:
gamma = 0.9
P_pi = np.array([[0, 1, 0],
                 [0, 0, 1],
                 [0, 0, 1]], dtype=float)   # переходы при политике «вправо»
R_pi = np.array([0.0, 1.0, 0.0])           # ожидаемая награда за шаг из s

V = np.zeros(3)
for it in range(30):
    V_new = R_pi + gamma * P_pi @ V
    V_new[2] = 0.0              # терминальное состояние
    if np.max(np.abs(V_new - V)) < 1e-10:
        break
    V = V_new
print(f"сошлось за {it} итераций: V = {V.round(4)}  (ожидаем [γ, 1, 0] = [{gamma}, 1, 0])")

## 8. Награда — это формулировка задачи, и её легко испортить

Агент оптимизирует ровно то, что написано в награде, а не то, что вы имели в виду. Классический пример — [CoastRunners](https://openai.com/index/faulty-reward-functions/): агента-лодку обучали на игровые очки, и вместо прохождения трассы он нашёл лагуну, где можно бесконечно крутиться, собирать бонусы и врезаться в стены. Очков больше, чем у любого честного игрока. Это называется **reward hacking**; обзор с десятками примеров есть у [Lilian Weng](https://lilianweng.github.io/posts/2024-11-28-reward-hacking/).

Практическое правило: награда должна описывать *что* нужно получить, а не *как* это делать. Если написать в награду «держи угол шеста маленьким», агент найдёт способ держать угол маленьким, необязательно тот, который вы ожидали.

## 9. Exploration vs exploitation: многорукие бандиты

Разберём задачу **без состояний**: она изолированно показывает главную дилемму RL.

Бытовые примеры:

* обед: пойти в проверенное кафе или попробовать новое?
* дорога на работу: знакомый маршрут или тот, что предлагает навигатор?
* A/B-тестирование: показывать баннер, который уже даёт клики, или тестировать новые?

### Постановка

* $K$ «рук» (действий); у руки $k$ своё неизвестное распределение награды со средним $\mu_k$.
* На шаге $t$ агент выбирает руку $a_t$ и получает награду $r_t \sim P(r \mid a_t)$.
* Цель — максимизировать суммарную награду за $T$ шагов, то есть минимизировать **regret**:

$$
\text{Regret}_T = T \cdot \mu^* - \mathbb{E}\Big[\sum_{t=1}^{T} r_t\Big], \qquad \mu^* = \max_k \mu_k .
$$

Агент не знает $\mu_k$ и оценивает их по ходу дела средним наблюдённых наград $\hat Q(a)$. Отсюда дилемма:

* **Exploitation** — выбирать руку с лучшей текущей оценкой. Хорошо *сейчас*.
* **Exploration** — пробовать другие руки, чтобы уточнить оценки. Стоит награды *сейчас*, окупается *потом*.

### Две простые стратегии

**ε-greedy.** С вероятностью $1-\varepsilon$ выбираем руку с максимальной оценкой $\hat Q(a)$, с вероятностью $\varepsilon$ — случайную. Просто и работает, но exploration «слепой»: агент одинаково часто трогает хорошо изученную плохую руку и плохо изученную.

**UCB1 (Upper Confidence Bound).** Выбираем руку по верхней доверительной границе:

$$
a_t = \arg\max_a \Big[ \hat{Q}(a) + c \sqrt{\frac{\ln t}{N(a)}} \Big],
$$

где $N(a)$ — сколько раз выбрали руку $a$. «Оптимизм перед лицом неопределённости»: чем меньше знаем про руку, тем больше бонус, и он убывает с накоплением статистики. Обе стратегии при разумных гиперпараметрах дают regret, растущий как $O(\log T)$; на семинаре реализуем и сравним.

Сначала посмотрим, что бывает **без** exploration.

In [ ]:
# Чисто жадная стратегия (exploration = 0) может навсегда «залипнуть» на плохой руке.
rng = np.random.default_rng(0)
true_means = [0.2, 0.5, 0.55]  # руки: плохая, средняя, лучшая

def pure_greedy_regret(n_steps=500, n_init=1):
    Q = np.zeros(len(true_means))
    N = np.zeros(len(true_means))
    regret = np.zeros(n_steps)
    for a in range(len(true_means)):          # инициализация: n_init проб каждой руки
        for _ in range(n_init):
            r = rng.binomial(1, true_means[a])
            N[a] += 1
            Q[a] += (r - Q[a]) / N[a]
    for t in range(n_steps):
        a = np.argmax(Q)
        r = rng.binomial(1, true_means[a])
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        regret[t] = max(true_means) - true_means[a]
    return np.cumsum(regret)

for n_init in [1, 5, 20]:
    for run in range(5):
        plt.plot(pure_greedy_regret(n_init=n_init), color=f"C{[1,5,20].index(n_init)}",
                 alpha=0.7, label=f"n_init={n_init}" if run == 0 else None)
plt.xlabel("шаг t")
plt.ylabel("суммарный regret")
plt.title("Чисто жадная стратегия: 5 запусков на каждую инициализацию")
plt.legend()
plt.show()

При малом числе начальных проб жадная стратегия с заметной вероятностью «залипает» на неоптимальной руке: regret растёт **линейно**, а не логарифмически. Чистый exploitation без exploration не гарантирует сходимости к оптимуму. В deep RL та же проблема выглядит как агент, который нашёл один способ получать маленькую награду и больше ничего не пробует.

## 10. Первый алгоритм: метод Cross-Entropy

Теперь у нас есть всё, чтобы написать первый настоящий RL-алгоритм. Идея простейшая, почти «эволюционная»:

1. Возьмём случайную политику $\pi_0$ (в каждом состоянии все действия равновероятны).
2. **Policy evaluation.** Сыграем $K$ сессий (эпизодов) по текущей политике $\pi_n$, получим траектории $\tau_k$ и их returns $G(\tau_k)$. Средний return оценивает качество политики (закон больших чисел, метод Монте-Карло):

$$
\mathbb{E}_{\pi_n}[G] \approx \frac{1}{K} \sum_{k=1}^{K} G(\tau_k).
$$

3. **Отбор элиты.** Посчитаем $q$-квантиль $\gamma_q$ returns (например, $q = 0.7$) и оставим только **элитные** траектории с $G(\tau_k) \ge \gamma_q$.

4. **Policy improvement.** Новая политика — частоты действий в элитных траекториях:

$$
\pi_{n+1}(a \mid s) = \frac{\big|\{(s, a) \in \text{элита}\}\big|}{\big|\{s \in \text{элита}\}\big|}.
$$

Если состояние в элите не встречалось, оставляем старое распределение.

5. Повторяем $N$ раз.

Название — от того, что шаг 4 минимизирует кросс-энтропию между новой политикой и распределением действий в элитных траекториях. Никакой «истинной» модели среды ($P$, $R$) алгоритм не знает: только играет и смотрит на результат. Это **model-free** метод.

**Проблемы** (и их решения, которые попробуете на семинаре):

* нужно много сессий;
* выбор политики сильно зависит от случайности: одна удачная траектория может «выключить» все остальные действия. Лечится **сглаживанием**: по Лапласу $\pi_{n+1}(a \mid s) = \frac{n(s,a) + \lambda}{n(s) + \lambda |\mathcal{A}|}$ или смешиванием со старой политикой $\pi_{n+1} \leftarrow \lambda \pi_{n+1} + (1-\lambda) \pi_n$;
* в стохастической среде хорошая траектория может быть просто везением;
* в таком виде работает только с конечными $\mathcal{S}$ и $\mathcal{A}$; с нейросетью вместо таблицы (неделя 5) ограничение снимается.

Запустим на Frozen Lake без скольжения.

In [ ]:
def run_session(env, policy, rng, max_steps=100):
    obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, total = [], [], 0.0
    for _ in range(max_steps):
        a = int(rng.choice(policy.shape[1], p=policy[obs]))
        states.append(obs); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return states, actions, total


def cross_entropy_method(env, n_iter=20, n_sessions=100, q=0.7, laplace=0.0, mix=1.0, seed=0):
    rng = np.random.default_rng(seed)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    policy = np.ones((n_states, n_actions)) / n_actions
    history = []
    for it in range(n_iter):
        sessions = [run_session(env, policy, rng) for _ in range(n_sessions)]
        returns = np.array([G for _, _, G in sessions])
        history.append(returns.mean())                       # policy evaluation
        threshold = np.quantile(returns, q)
        elite = [s for s in sessions if s[2] >= threshold and s[2] > 0]
        counts = np.full((n_states, n_actions), laplace)
        for states, actions, _ in elite:                     # policy improvement
            for s, a in zip(states, actions):
                counts[s, a] += 1
        new_policy = policy.copy()
        seen = counts.sum(axis=1) > 0
        new_policy[seen] = counts[seen] / counts[seen].sum(axis=1, keepdims=True)
        policy = mix * new_policy + (1 - mix) * policy
    return policy, history


env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)
policy_plain, hist_plain = cross_entropy_method(env)
policy_smooth, hist_smooth = cross_entropy_method(env, laplace=0.5, mix=0.5)

plt.plot(hist_plain, marker="o", label="без сглаживания")
plt.plot(hist_smooth, marker="o", label="сглаживание: Лаплас λ=0.5, смесь 0.5")
plt.xlabel("итерация")
plt.ylabel("средний return за итерацию (доля успехов)")
plt.title("Cross-Entropy на FrozenLake 4x4 без скольжения")
plt.legend()
plt.show()

arrows = "←↓→↑"
print("выученная политика (стрелка = самое вероятное действие):")
for row in range(4):
    print("  " + " ".join(arrows[int(np.argmax(policy_smooth[row * 4 + col]))] for col in range(4)))

За несколько итераций из случайной политики получается маршрут до цели. Без сглаживания алгоритм «схлопывается» в первую же успешную траекторию (в детерминированной среде это работает, в скользкой — нет), со сглаживанием учится плавнее. На семинаре реализуете этот метод сами и посмотрите, что будет на скользком льду; на неделе 5 заменим таблицу нейросетью и решим CartPole.

## 11. Где применяется RL

Общий рецепт: как только у задачи есть **среда** (или её симулятор) и **числовая цель**, поведение можно не программировать, а выучить.

![timeline](../../assets/rl_timeline.png)

### Игры

| Год | Система | Что произошло | Посмотреть |
|---|---|---|---|
| 1992 | **TD-Gammon** | Нейросеть + TD-обучение играет в нарды на уровне чемпионов мира. | [статья](https://dl.acm.org/doi/10.1145/203330.203343) |
| 2015 | **DQN** | Одна сеть учится играть в 49 игр Atari по пикселям и счёту. | [видео Breakout](https://www.youtube.com/watch?v=TmPfTpjtdgg), [Nature](https://www.nature.com/articles/nature14236) |
| 2016 | **AlphaGo** | Победа над Ли Седолем в го. | [фильм](https://www.youtube.com/watch?v=WXuK6gekU1Y) |
| 2017 | **AlphaZero** | Шахматы, сёги и го с нуля, только игрой с самим собой. | [блог](https://deepmind.google/discover/blog/alphazero-shedding-new-light-on-chess-shogi-and-go/) |
| 2019 | **OpenAI Five**, **AlphaStar** | Dota 2 и StarCraft II на уровне профессионалов. | [OpenAI Five](https://openai.com/index/openai-five/), [AlphaStar](https://deepmind.google/discover/blog/alphastar-mastering-the-real-time-strategy-game-starcraft-ii/) |
| 2019 | **Hide and Seek** | Агенты в прятках сами изобретают использование инструментов. | [блог с гифками](https://openai.com/index/emergent-tool-use/) |

Почему игры: есть симулятор, чёткая награда и можно сыграть миллионы партий.

### Робототехника

[Роботы-футболисты DeepMind](https://sites.google.com/view/op3-soccer), [ходьба четвероногих ANYmal](https://arxiv.org/abs/1901.08652), [Boston Dynamics Spot](https://bostondynamics.com/blog/starting-on-the-right-foot-with-reinforcement-learning/), [кубик Рубика одной рукой](https://openai.com/index/solving-rubiks-cube/). Общая схема: учим в симуляторе, переносим на железо (sim-to-real).

### Управление инфраструктурой

* [Охлаждение дата-центров Google](https://deepmind.google/discover/blog/deepmind-ai-reduces-google-data-centre-cooling-bill-by-40/): минус 40% энергии на охлаждение. Состояние — сотни датчиков, действия — уставки оборудования, награда — энергия при соблюдении температурных ограничений.
* [Управление плазмой в токамаке](https://www.nature.com/articles/s41586-021-04301-9): RL-контроллер держит форму плазмы через токи в 19 магнитных катушках.
* Светофоры, балансировка сетей, [размещение блоков на чипе](https://deepmind.google/discover/blog/how-alphachip-transformed-computer-chip-design/).

### Финансы: торговля на бирже

Исполнение крупных ордеров (как разбить заявку, чтобы не сдвинуть цену), маркет-мейкинг, управление портфелем. Честная оговорка: рынок нестационарен и очень шумный, поэтому это гораздо труднее, чем Atari. Обзор: [Deep RL for trading](https://arxiv.org/abs/1911.10107), учебная библиотека [FinRL](https://github.com/AI4Finance-Foundation/FinRL).

### Роевое поведение дронов

Несколько агентов учатся одновременно держать строй, облетать препятствия, вместе искать цель. Каждый дрон видит только соседей, награда общая — это **multi-agent RL** (неделя 15). [Полёт роя сквозь лес](https://arxiv.org/abs/2202.03308), среды [PettingZoo](https://pettingzoo.farama.org/) и [gym-pybullet-drones](https://github.com/utiasDSL/gym-pybullet-drones).

### Рекомендации и языковые модели

Что показать пользователю, чтобы он остался надолго, а не только кликнул сейчас. **RLHF** ([InstructGPT](https://arxiv.org/abs/2203.02155)): ChatGPT стал полезным собеседником благодаря RL на человеческих предпочтениях; **рассуждающие модели** ([DeepSeek-R1](https://arxiv.org/abs/2501.12948)) обучены RL на проверяемых наградах.

**Интерактивные демо**: [ReinforceJS](https://cs.stanford.edu/people/karpathy/reinforcejs/) (GridWorld в браузере), [каталог сред Gymnasium](https://gymnasium.farama.org/environments/classic_control/), [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction).

## 12. Карта курса

![course](../../assets/course_map.png)

Первые четыре недели — **основы**: постановка задачи, построение сред, табличные value-based и policy-based методы. Дальше те же идеи переносим на нейросети (**deep RL**: DQN, policy gradient, actor-critic), потом продвинутые методы (PPO, TD3, model-based), и расширения: иерархический и многоагентный RL, трансформеры.

![taxonomy](../../assets/rl_taxonomy.png)

Всё, что в этой таблице, мы **реализуем сами** с нуля: от Cross-Entropy до PPO и TD3. RL-алгоритмы печально известны тем, что «почти работающая» реализация не работает вовсе, и понять, где ошибка, можно только зная каждую строчку.

## Итоги лекции

1. **Reinforcement Learning зарождался в психологии.** «Подкрепление» и закон эффекта описывают тот же механизм, который мы теперь программируем; математику к нему добавила теория управления.
2. **Reinforcement Learning можно выделить в отдельную группу алгоритмов машинного обучения.** Нет правильных ответов, награда отложена, данные порождает сам агент, нужно исследовать. Ни одной из этих черт нет в supervised и unsupervised learning.
3. **Reinforcement Learning широко применяется в самых разных областях: от игр до управления охлаждением дата-центров и от торговли на бирже до роевого поведения дронов.** Рецепт один: среда + числовая цель.

Словарь, который нужно унести с собой: **агент, среда, состояние, действие, награда, политика, эпизод, return, марковское свойство, MDP, ценность $V$ и $Q$, уравнение Беллмана, exploration и exploitation**.

## Литература

* R. Sutton, A. Barto. *Reinforcement Learning: An Introduction*, 2nd ed. — [бесплатный PDF](http://incompleteideas.net/book/the-book-2nd.html). Главы 1–3 покрывают сегодняшнюю лекцию.
* D. Silver. [UCL Course on RL](https://www.davidsilver.uk/teaching/), лекции 1–2: введение и MDP.
* А. Плаксин. [Курс imm-rl-lab](https://github.com/imm-rl-lab/reinforcement_learning_course), лекция 1: введение и метод Cross-Entropy.
* [OpenAI Spinning Up](https://spinningup.openai.com/) — короткое введение + чистые реализации.
* [Gymnasium](https://gymnasium.farama.org/) — документация по средам и интерфейсу.

## На семинаре и дома

* Семинар (`../seminar/seminar.ipynb`): интерфейс Gymnasium на Frozen Lake, политика-таблица, среда бандита и агенты ε-greedy / UCB1, реализация Cross-Entropy.
* Мини-семинар по PyTorch (`../seminar/pytorch_intro.ipynb`) для тех, кто с ним не работал.
* ДЗ (`../homework/homework.ipynb`): бандиты, теория (return, вывод Беллмана, марковская цепь, MDP на бумаге), Cross-Entropy на Frozen Lake 8×8.